# Deep Learning Model Experiments
**CNN + LSTM + GRU × 7 source configs = 21 experiments**

Requires GPU runtime. Estimated time: ~2-3 hours on T4.

Place the `data/splits/` folder under `/content/data/splits`.

## 1. Setup

In [1]:
# === Path Configuration ===
# Place the data/splits folder under /content/data/splits
SPLITS_DIR = "/content/data/splits"
RESULTS_DIR = "/content/results"
MODELS_DIR = "/content/models"

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# Verify data
files = os.listdir(SPLITS_DIR)
print(f"Found {len(files)} split files")
assert len(files) >= 21, f"Expected 21 split files, found {len(files)}"

Found 22 split files


In [2]:
import csv
import random
from collections import Counter
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
)

# ====== Config ======
SEED = 42
LABEL_MAP = {"negative": 0, "neutral": 1, "positive": 2}
LABEL_NAMES = ["negative", "neutral", "positive"]

EMBEDDING_DIM = 300
DL_HIDDEN_DIM = 128
DL_DROPOUT = 0.3
DL_BATCH_SIZE = 64
DL_MAX_EPOCHS = 20
DL_PATIENCE = 3
DL_LEARNING_RATE = 1e-3
MAX_VOCAB_SIZE = 50000
MAX_SEQ_LENGTH_TWEET = 64
MAX_SEQ_LENGTH_REVIEW = 256

ALL_CONFIGS = [
    "twitter", "skytrax", "airlinequality",
    "twitter+skytrax", "twitter+airlinequality", "skytrax+airlinequality",
    "twitter+skytrax+airlinequality"
]

# Reproducibility
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed()

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA A100-SXM4-80GB


## 2. Data Utilities

In [3]:
PAD_IDX = 0
UNK_IDX = 1


def load_split(source_config, split):
    path = os.path.join(SPLITS_DIR, f"{source_config}_{split}.csv")
    texts, labels = [], []
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            texts.append(row["text"])
            labels.append(LABEL_MAP[row["label"]])
    return texts, labels


def get_max_seq_length(source_config):
    return MAX_SEQ_LENGTH_TWEET if source_config == "twitter" else MAX_SEQ_LENGTH_REVIEW


def build_vocab(texts, max_size=MAX_VOCAB_SIZE):
    counter = Counter()
    for text in texts:
        counter.update(text.split())
    most_common = counter.most_common(max_size - 2)
    word2idx = {"<PAD>": PAD_IDX, "<UNK>": UNK_IDX}
    for word, _ in most_common:
        word2idx[word] = len(word2idx)
    return word2idx


def texts_to_sequences(texts, word2idx, max_len):
    sequences = np.zeros((len(texts), max_len), dtype=np.int64)
    for i, text in enumerate(texts):
        tokens = text.split()[:max_len]
        for j, token in enumerate(tokens):
            sequences[i, j] = word2idx.get(token, UNK_IDX)
    return sequences


def compute_class_weights(labels, num_classes=3):
    counts = Counter(labels)
    total = len(labels)
    return [total / (num_classes * counts[c]) if counts[c] > 0 else 1.0 for c in range(num_classes)]


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "macro_precision": precision_score(y_true, y_pred, average="macro"),
        "macro_recall": recall_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }


def compute_per_class_metrics(y_true, y_pred):
    result = {}
    for avg in ["macro", "weighted", None]:
        p = precision_score(y_true, y_pred, average=avg, zero_division=0)
        r = recall_score(y_true, y_pred, average=avg, zero_division=0)
        f = f1_score(y_true, y_pred, average=avg, zero_division=0)
        if avg is None:
            for i, name in enumerate(LABEL_NAMES):
                result[f"{name}_precision"] = p[i]
                result[f"{name}_recall"] = r[i]
                result[f"{name}_f1"] = f[i]
        else:
            result[f"{avg}_precision"] = p
            result[f"{avg}_recall"] = r
            result[f"{avg}_f1"] = f
    return result

print("Data utilities loaded.")

Data utilities loaded.


## 3. Model Definitions

In [4]:
class TextCNN(nn.Module):
    """Multi-kernel CNN text classifier."""
    def __init__(self, vocab_size, embed_dim, num_classes, num_filters=128,
                 kernel_sizes=(3, 4, 5), dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, k) for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def forward(self, x):
        x = self.embedding(x).permute(0, 2, 1)
        conv_outs = [torch.relu(conv(x)).max(dim=2).values for conv in self.convs]
        x = torch.cat(conv_outs, dim=1)
        return self.fc(self.dropout(x))


class TextLSTM(nn.Module):
    """Bidirectional LSTM text classifier."""
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        hidden = torch.cat((hidden[0], hidden[1]), dim=1)
        return self.fc(self.dropout(hidden))


class TextGRU(nn.Module):
    """Bidirectional GRU text classifier."""
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.embedding(x)
        _, hidden = self.gru(x)
        hidden = torch.cat((hidden[0], hidden[1]), dim=1)
        return self.fc(self.dropout(hidden))


MODEL_REGISTRY = {"cnn": TextCNN, "lstm": TextLSTM, "gru": TextGRU}
print("Models defined.")

Models defined.


## 4. Training Loop

In [5]:
def create_model(model_key, vocab_size):
    if model_key == "cnn":
        return TextCNN(vocab_size, EMBEDDING_DIM, 3, num_filters=DL_HIDDEN_DIM,
                       kernel_sizes=(3, 4, 5), dropout=0.5)
    elif model_key == "lstm":
        return TextLSTM(vocab_size, EMBEDDING_DIM, DL_HIDDEN_DIM, 3, dropout=DL_DROPOUT)
    else:
        return TextGRU(vocab_size, EMBEDDING_DIM, DL_HIDDEN_DIM, 3, dropout=DL_DROPOUT)


def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
        correct += (logits.argmax(1) == y_batch).sum().item()
        total += len(y_batch)
    return total_loss / total, correct / total


def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            total_loss += loss.item() * len(y_batch)
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    return total_loss / len(all_labels), np.array(all_preds), np.array(all_labels)


def run_experiment(model_key, source_config):
    """Train and evaluate one DL model on one source config."""
    set_seed()
    print(f"\n{'='*60}")
    print(f"[TRAIN] {model_key.upper()} | {source_config}")
    print(f"{'='*60}")

    # Load data
    train_texts, train_labels = load_split(source_config, "train")
    val_texts, val_labels = load_split(source_config, "val")
    test_texts, test_labels = load_split(source_config, "test")

    max_len = get_max_seq_length(source_config)
    word2idx = build_vocab(train_texts)
    vocab_size = len(word2idx)

    X_train = texts_to_sequences(train_texts, word2idx, max_len)
    X_val = texts_to_sequences(val_texts, word2idx, max_len)
    X_test = texts_to_sequences(test_texts, word2idx, max_len)

    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_train), torch.tensor(train_labels)),
        batch_size=DL_BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_val), torch.tensor(val_labels)),
        batch_size=DL_BATCH_SIZE)
    test_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_test), torch.tensor(test_labels)),
        batch_size=DL_BATCH_SIZE)

    model = create_model(model_key, vocab_size).to(device)

    weights = compute_class_weights(train_labels)
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(weights, dtype=torch.float32).to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=DL_LEARNING_RATE)

    best_val_loss = float("inf")
    patience_counter = 0
    best_state = None

    for epoch in range(1, DL_MAX_EPOCHS + 1):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_preds, val_labels_arr = evaluate_model(model, val_loader, criterion)
        val_f1 = f1_score(val_labels_arr, val_preds, average="macro")

        print(f"  Epoch {epoch:2d}/{DL_MAX_EPOCHS} | "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} val_macro_f1={val_f1:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= DL_PATIENCE:
                print(f"  [EARLY STOP] No improvement for {DL_PATIENCE} epochs.")
                break

    # Restore best and evaluate on test
    model.load_state_dict(best_state)
    model = model.to(device)
    _, test_preds, test_labels_arr = evaluate_model(model, test_loader, criterion)

    metrics = compute_metrics(test_labels_arr, test_preds)
    per_class = compute_per_class_metrics(test_labels_arr, test_preds)
    cm = confusion_matrix(test_labels_arr, test_preds)

    print(f"  TEST | Accuracy: {metrics['accuracy']:.4f} | Macro-F1: {metrics['macro_f1']:.4f}")
    print(f"  Confusion Matrix:\n{cm}")

    # Save model
    save_dir = os.path.join(MODELS_DIR, model_key, source_config)
    os.makedirs(save_dir, exist_ok=True)
    torch.save({
        "model_state_dict": model.cpu().state_dict(),
        "word2idx": word2idx,
        "max_len": max_len,
        "model_key": model_key,
        "vocab_size": vocab_size,
    }, os.path.join(save_dir, "model.pt"))

    return {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "model": model_key,
        "source_config": source_config,
        **metrics, **per_class,
    }

print("Training loop defined.")

Training loop defined.


## 5. Run All 21 Experiments

Results are saved after each experiment for crash recovery.

In [6]:
all_results = []

for model_key in ["cnn", "lstm", "gru"]:
    for source_config in ALL_CONFIGS:
        result = run_experiment(model_key, source_config)
        all_results.append(result)

        # Save after each experiment (crash recovery)
        path = os.path.join(RESULTS_DIR, "experiment_log_dl.csv")
        with open(path, "w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(all_results[0].keys()))
            writer.writeheader()
            writer.writerows(all_results)
        print(f"  [{len(all_results)}/21] Progress saved.")

print(f"\n{'='*60}")
print(f"DONE: {len(all_results)} experiments completed.")


[TRAIN] CNN | twitter
  Epoch  1/20 | train_loss=0.9723 train_acc=0.5511 | val_loss=0.7948 val_macro_f1=0.6197
  Epoch  2/20 | train_loss=0.6450 train_acc=0.7326 | val_loss=0.7999 val_macro_f1=0.6324
  Epoch  3/20 | train_loss=0.4625 train_acc=0.8076 | val_loss=0.8547 val_macro_f1=0.6657
  Epoch  4/20 | train_loss=0.3314 train_acc=0.8636 | val_loss=0.8748 val_macro_f1=0.6543
  [EARLY STOP] No improvement for 3 epochs.
  TEST | Accuracy: 0.6743 | Macro-F1: 0.6266
  Confusion Matrix:
[[935 334  84]
 [ 90 294  45]
 [ 54  79 191]]
  [1/21] Progress saved.

[TRAIN] CNN | skytrax
  Epoch  1/20 | train_loss=1.0313 train_acc=0.5679 | val_loss=0.7750 val_macro_f1=0.5988
  Epoch  2/20 | train_loss=0.8365 train_acc=0.6713 | val_loss=0.7433 val_macro_f1=0.6418
  Epoch  3/20 | train_loss=0.7550 train_acc=0.7129 | val_loss=0.7408 val_macro_f1=0.6089
  Epoch  4/20 | train_loss=0.6757 train_acc=0.7478 | val_loss=0.7662 val_macro_f1=0.6526
  Epoch  5/20 | train_loss=0.5690 train_acc=0.7941 | val_loss=

## 6. Results Summary

In [7]:
print(f"{'Model':<8} {'Source Config':<35} {'Accuracy':>10} {'Macro-F1':>10}")
print("=" * 68)
for r in all_results:
    print(f"{r['model']:<8} {r['source_config']:<35} {r['accuracy']:>10.4f} {r['macro_f1']:>10.4f}")

best = max(all_results, key=lambda x: x["macro_f1"])
print(f"\nBest: {best['model']} | {best['source_config']} | Macro-F1: {best['macro_f1']:.4f}")

Model    Source Config                         Accuracy   Macro-F1
cnn      twitter                                 0.6743     0.6266
cnn      skytrax                                 0.7579     0.6683
cnn      airlinequality                             0.7242     0.5738
cnn      twitter+skytrax                         0.7824     0.6919
cnn      twitter+airlinequality                     0.6983     0.6201
cnn      skytrax+airlinequality                     0.7494     0.6591
cnn      twitter+skytrax+airlinequality             0.7281     0.6666
lstm     twitter                                 0.7089     0.6476
lstm     skytrax                                 0.7264     0.6168
lstm     airlinequality                             0.6705     0.5001
lstm     twitter+skytrax                         0.7232     0.6657
lstm     twitter+airlinequality                     0.7178     0.6201
lstm     skytrax+airlinequality                     0.7204     0.6291
lstm     twitter+skytrax+airlinequality  

## 7. Download Results

In [9]:
import subprocess
#subprocess.run(["zip", "-r", "/content/results_dl.zip", "/content/results", "/content/models"], check=True)
subprocess.run(["zip", "-r", "/content/models_dl_cnn.zip", "/content/models/cnn"], check=True)
subprocess.run(["zip", "-r", "/content/models_dl_lstm.zip", "/content/models/lstm"], check=True)
subprocess.run(["zip", "-r", "/content/models_dl_gru.zip", "/content/models/gru"], check=True)
subprocess.run(["zip", "-r", "/content/results_dl.zip", "/content/results"], check=True)
#print("Zip created: /content/models_dl_cnn.zip")
print("Download this file from the file explorer.")

Download this file from the file explorer.
